In [1]:
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
CODE_DIR = os.environ.get('AGES_CODE', '/content/drive/MyDrive/ages-resolution-time/code')
sys.path.insert(0, CODE_DIR); os.chdir(CODE_DIR)
sys.argv = ['notebook'] + ['a', 'b', 'c', 'd']


Mounted at /content/drive


In [2]:
# Analiză de sensibilitate: Protocol B vs C doar cu atribute disponibile la crearea issue-ului
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, matthews_corrcoef
from ages_common import (RANDOM_STATE, TARGET, PROJECT, MODEL_ORDER, ensure, load_prepared,
                         baseline_models, cv_random, cv_chronological, select_threshold_oof)

OUT = ensure(f'{PROJECT}/11_creation_time_features')
df, feature_cols = load_prepared()

CREATION = ['created_hour', 'created_dayofweek', 'title_len', 'log_body_len',
            'n_complexity_words', 'n_urgency_words', 'n_negative_words', 'body_has_steps']
EXCLUDED = [c for c in feature_cols if c.startswith(('assignee_', 'comp_', 'cat_'))
            or c in ('n_assignees', 'n_labels', 'is_feature_label', 'has_milestone')]
FEATURE_SETS = {
    'All features': feature_cols,
    'Without assignee, labels, milestone, category and components': [c for c in feature_cols if c not in EXCLUDED],
    'Title, body and creation time only': CREATION,
}

d = df.sort_values('Created At').reset_index(drop=True)
n = len(d); k = int(n * 0.8); n_test = n - k
y = d[TARGET]

def evaluate(X_tr, y_tr, X_te, y_te, cv):
    res = {}
    for name in MODEL_ORDER:
        model = baseline_models()[name]
        thr = select_threshold_oof(model, X_tr, y_tr, cv, verbose=False)
        proba = clone(model).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
        res[name] = (roc_auc_score(y_te, proba), matthews_corrcoef(y_te, (proba >= thr).astype(int)))
    return pd.DataFrame(res, index=['AUC', 'MCC']).T

rows = []
for label, cols in FEATURE_SETS.items():
    X = d[cols]
    C = evaluate(X.iloc[:k], y.iloc[:k], X.iloc[k:], y.iloc[k:], cv_chronological())
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=n_test, stratify=y, random_state=RANDOM_STATE)
    B = evaluate(X_tr, y_tr, X_te, y_te, cv_random())
    dA, dM = C['AUC'] - B['AUC'], C['MCC'] - B['MCC']
    rows.append({
        'Feature set': label,
        'Features': len(cols),
        'Mean Test-AUC B': round(B['AUC'].mean(), 3),
        'Mean Test-AUC C': round(C['AUC'].mean(), 3),
        'Mean dAUC C-B (min, max)': f'{dA.mean():.3f} ({dA.min():.3f}, {dA.max():.3f})',
        'Mean MCC B': round(B['MCC'].mean(), 3),
        'Mean MCC C': round(C['MCC'].mean(), 3),
        'Mean dMCC C-B (min, max)': f'{dM.mean():.3f} ({dM.min():.3f}, {dM.max():.3f})',
        'Models with lower Test-AUC under C': f'{(dA < 0).sum()}/7',
        'Models with lower MCC under C': f'{(dM < 0).sum()}/7',
    })
    print(f'{label}: done')

table = pd.DataFrame(rows)
table.to_csv(f'{OUT}/table_creation_time_features.csv', index=False)
table

All features: done
Without assignee, labels, milestone, category and components: done
Title, body and creation time only: done


,Feature set,Features,Mean Test-AUC B,Mean Test-AUC C,"Mean dAUC C-B (min, max)",Mean MCC B,Mean MCC C,"Mean dMCC C-B (min, max)",Models with lower Test-AUC under C,Models with lower MCC under C
0,All features,27,0.693,0.667,"-0.026 (-0.057, 0.020)",0.268,0.218,"-0.050 (-0.121, 0.042)",6/7,6/7
1,"Without assignee, labels, milestone, category ...",9,0.631,0.590,"-0.041 (-0.087, 0.001)",0.161,0.062,"-0.099 (-0.164, -0.051)",6/7,7/7
2,"Title, body and creation time only",8,0.595,0.584,"-0.011 (-0.057, 0.019)",0.073,0.006,"-0.066 (-0.113, -0.003)",3/7,7/7
